<a href="https://colab.research.google.com/github/elle-et-noire/lambda-talk/blob/main/notebooks/lambda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lambda Calculus in Python：演習ノートブック

講義スライド「Lambda Calculus in Python」の内容を、
Google Colab 上で動かすためのノートブックです。

- Colab：[https://colab.research.google.com/github/elle-et-noire/lambda-talk/blob/main/notebooks/lambda.ipynb](https://colab.research.google.com/github/elle-et-noire/lambda-talk/blob/main/notebooks/lambda.ipynb)
- リポジトリ（スライド・参照実装）：[https://github.com/elle-et-noire/lambda-talk](https://github.com/elle-et-noire/lambda-talk)

**§番号＝スライドの章番号** で（§3 ＝ 3 章「チャーチ真偽値」）、
各小見出しの「スライド：…」は対応するスライドのタイトルです。

## §0 使い方

- 前のセルの定義を後のセルが使うので、**上から順に** `Shift + Enter` で実行します。
  途中から始めるときは「ランタイム → 以前のセルを実行」を使います。
- わざとエラー（`RecursionError` など）を見せるセルは `try` / `except` で包んであってエラー名を表示するだけなので、
  「ランタイム → すべてのセルを実行」でも最後までエラーなく通ります。
- **変更例**：数や引数を書き換えて、結果の変化を確かめるためのセルです。
- **演習**：`MY_OR = None` のような行を自分の定義に書き換えて実行すると、答え合わせが出ます。
- **解答** セルは Colab では折りたたまれているので（タイトルだけ表示）、自分で考えてから、ダブルクリックで開いてください。
  解答セルは後の節で使う正しい定義も読み込むので、**開かなくても実行だけはしてください**（すべて実行なら自動で実行されます）。
- 外部パッケージは使わず、Python 3.10 以上で動きます。リポジトリの `code/lambda_core.py` は読み込まず、必要な定義はすべてこのノートブックの中に書いてあります。

### 準備

このノートブックでは、答え合わせ用の `check` や、実験用に数を作る `to_church` などの
道具（数値リテラルなども使う普通の Python）を使うので、下のセルで準備します。

In [ ]:
import sys
import time

print("Python", sys.version.split()[0])


def check(candidate, tests):
    """演習の答え合わせ用の道具（数値リテラルなども使う普通の Python）。

    tests は (説明, 候補を受け取って結果を返す関数, 期待値) のリスト。
    """
    if candidate is None:
        print("まだ None のままです。定義を書いてから実行してください。")
        return
    ok = 0
    for label, run, expected in tests:
        try:
            got = run(candidate)
        except Exception as error:  # RecursionError なども表示だけする
            got = type(error).__name__
        ok += got == expected
        mark = "OK" if got == expected else "NG"
        print(f"{mark}  {label} -> {got!r}（期待値 {expected!r}）")
    print(f"{ok}/{len(tests)} 件正解")

## §1 導入（スライド 1 章）

1 章は導入（決定問題と Church–Turing thesis）でコードはなく、ここには 1 章の最後のスライドで説明するこの講義の書き方をまとめます。λ記法と Python の lambda の対応は §2 で扱います。

### lambda と関数呼び出しだけで書く（スライド：「lambda と関数呼び出しだけで書く」）

ラムダ計算は変数・抽象・適用の 3 種類の項だけで計算を定義していて、Python ではそれぞれ変数・無名関数（lambda 式）・関数呼び出しに当たります。そこで Python でも、1 引数の `lambda` 式と関数呼び出し `f(x)`、式に付ける名前（`ADD = ...`）だけで書きます。名前が出てくるところは、定義の右辺に置き換えられます。
Python にある数値リテラル、`True`・`False`、`if`・`while`、`def`・`class`、`+` や `==` などの演算子は使わず、真偽値・数・分岐・再帰を関数で作ります。

例外は、結果を Python の値に変換する関数（`to_bool`、`to_int`、`to_list`、`to_str` など `to_` で始まる関数）です。
変換関数は数値リテラルや `+` も使う普通の Python で書きますが、計算の途中では使わず、結果を Python の値に変換して表示するためだけに使います。

## §2 文法と評価（スライド 2 章）

### λ記法と恒等関数 `ID`（スライド：「λ記法と Python の lambda：項は 3 種類」）

| 項 | λ記法 | Python |
|---|---|---|
| 変数 | $x$ | `x` |
| 抽象 | $\lambda x.\,M$ | 無名関数 `lambda x: M` |
| 適用 | $M\,N$ | 関数呼び出し `M(N)` |

$\lambda$ → `lambda`、ドット → コロン、並べて書く → 括弧で呼ぶ。
読み方の約束として、$\lambda x.\,M\,N$ の本体は $M\,N$ 全体で、Python の `lambda x: M(N)` に当たります。
文字列 `"a"`、`"b"` は結果を見るためだけに渡し、定義の中には使いません。

In [ ]:
# lambda で作った値は「関数」としか表示されない
print(lambda x: x)
print((lambda x: x)(lambda y: y))

In [ ]:
ID = lambda x: x

print(repr(ID("a")))
print(repr(ID(ID)("b")))

### カリー化（スライド：「カリー化：2 引数の関数を `f(a)(b)` で呼ぶ」）

`lambda x, y: x` の代わりに `lambda x: lambda y: x` と書くと、`f(a)` は残りの引数 `y` を受け取る関数を返します。

In [ ]:
half = (lambda x: lambda y: x)("a")   # まだ y を待っている関数
print(half)
print(repr(half("b")))

In [ ]:
# 変更例：2 つ目を返す関数にする。"b" を別の値にする
print(repr((lambda x: lambda y: y)("a")("b")))

### 束縛変数・自由変数とβ簡約（スライド：「束縛変数と自由変数：引数と外側の名前」「β簡約：関数呼び出しで引数が本体に代入される」）

内側の `lambda x: (x, y)` では、引数 `x` が束縛変数、外側で決まる `y` が自由変数です。
関数を呼ぶと本体の引数に値が代入され（β簡約）、内側の `lambda x:` の `x` は内側の引数を指します。

In [ ]:
print((lambda y: lambda x: (x, y))("外")("引数"))

print(repr((lambda x: x(x))(ID)("a")))          # ID(ID)("a") と同じ
print(repr((lambda x: lambda x: x)("a")("b")))  # 内側の x は "b"

### α変換（スライド：「α変換：代入の前に引数名を付け替える」）

$(\lambda x.\,\lambda y.\,x)\;y$ の本体の `x` を文字どおり `y` に置き換えると $\lambda y.\,y$ になり、外側の `y` と内側の引数 `y` を取り違えます。
先に内側の `y` を `w` に付け替えてから代入すると、正しい結果 $\lambda w.\,y$ になります。

In [ ]:
print(repr((lambda y: (lambda x: lambda y: x)(y))("外")("内")))  # Python の結果
print(repr((lambda y: lambda y: y)("外")("内")))                 # 文字どおり代入
print(repr((lambda y: lambda w: y)("外")("内")))                 # y を w に付け替え

### 正格評価（スライド：「Python は引数を先に評価する：正格評価」）

Python は関数を呼ぶ前に、使わない引数でも先に評価しますが、**lambda の本体は呼ばれるまで評価しない** ので、`lambda _:` で包むと評価を遅らせられます。

In [ ]:
print(repr((lambda x: "a")(print("引数を計算"))))

# lambda _: で包むと、使われない引数の中身は評価されない
print(repr((lambda x: "a")(lambda _: print("これは表示されない"))))

### 発展：η変換と評価順序（スライド：「η変換：`lambda v: f(v)` と `f` は同じ関数」「評価順序：正規順序と適用順序」）

`lambda v: f(v)` と `f` は呼べば同じ結果になります（η変換）が、`lambda v: M(v)` は呼ばれるまで `M` を評価しません（6 章の Z で使います）。
Python は引数を先に評価するので適用順序に近く、使わない引数に止まらない式を渡すと RecursionError になります。

In [ ]:
print((lambda v: len(v))("abc"), len("abc"))   # η変換：同じ結果

try:
    (lambda x: "a")((lambda x: x(x))(lambda x: x(x)))
except RecursionError as error:
    print("RecursionError:", error)

# lambda で包めば、引数の中身は評価されない
print(repr((lambda x: "a")(lambda v: (lambda x: x(x))(lambda x: x(x))(v))))

## §3 チャーチ真偽値（スライド 3 章）

### `TRUE` と `FALSE`（スライド：「`TRUE` と `FALSE`：2 つの値から 1 つを選ぶ関数」）

真は 2 つの値を受け取り 1 つ目を返す関数、偽は 2 つ目を返す関数で、`TRUE(a)(b)` は `a if True else b` に当たります。

In [ ]:
TRUE = lambda t: lambda f: t
FALSE = lambda t: lambda f: f

print(repr(TRUE("then 側")("else 側")))
print(repr(FALSE("then 側")("else 側")))
print(TRUE)   # そのまま表示しても中身は見えない

### 変換関数 `to_bool`（スライド：「`TRUE` と `FALSE`：2 つの値から 1 つを選ぶ関数」）

In [ ]:
to_bool = lambda b: b(True)(False)

print(to_bool(TRUE), to_bool(FALSE))

### `IF`、`NOT`、`AND`（スライド：「`IF`：`IF(b)(x)(y)` は `x if b else y` に当たる」「`NOT` と `AND`：真偽値に渡す候補を工夫する」）

In [ ]:
IF = lambda b: lambda t: lambda f: b(t)(f)
NOT = lambda b: b(FALSE)(TRUE)
AND = lambda a: lambda b: a(b)(FALSE)

print(repr(IF(TRUE)("a")("b")), repr(IF(FALSE)("a")("b")))
print(to_bool(NOT(TRUE)))
print(to_bool(AND(TRUE)(TRUE)), to_bool(AND(TRUE)(FALSE)))

In [ ]:
# 変更例：真理値表を全部出す（AND は NOT や自作の演算に替えられる）
for a in [TRUE, FALSE]:
    for b in [TRUE, FALSE]:
        print(to_bool(a), to_bool(b), "->", to_bool(AND(a)(b)))

### 演習：`OR` を書く（スライド：「`OR` を書く」）

`MY_OR = lambda a: lambda b: ＿＿＿` を完成させよ。
ヒント：AND と同じく a に 2 択を選ばせるとして、2 つの候補は何か。

In [ ]:
MY_OR = None   # ← lambda a: lambda b: ... に書き換える

check(MY_OR, [
    ("OR(TRUE)(TRUE)", lambda OR: to_bool(OR(TRUE)(TRUE)), True),
    ("OR(TRUE)(FALSE)", lambda OR: to_bool(OR(TRUE)(FALSE)), True),
    ("OR(FALSE)(TRUE)", lambda OR: to_bool(OR(FALSE)(TRUE)), True),
    ("OR(FALSE)(FALSE)", lambda OR: to_bool(OR(FALSE)(FALSE)), False),
])

In [ ]:
#@title 解答：OR（スライド「`OR` の解答」）（自分で考えてから開く）

OR = lambda a: lambda b: a(TRUE)(b)

print(to_bool(OR(FALSE)(TRUE)), to_bool(OR(FALSE)(FALSE)))

### `IF` が両方の枝を評価する問題とサンク（スライド：「正格評価のため `IF` の枝をサンクで包む」）

2 章で見たとおり、Python は関数を呼ぶ **前に** 引数をすべて評価する（正格評価）ので、
else 側に止まらない式 `(lambda x: x(x))(lambda x: x(x))` を置くと、then が選ばれるはずでも失敗します（エラーを見せるセルなので `try` で包んであります）。

In [ ]:
try:
    to_bool(IF(TRUE)(TRUE)(
        (lambda x: x(x))(lambda x: x(x))))
except RecursionError as error:
    print("RecursionError:", error)

In [ ]:
# 対策：両方の枝を lambda _: で包み、選んだサンクに (ID) を渡して呼ぶ
print(to_bool(IF(TRUE)(lambda _: TRUE)(
    lambda _: (lambda x: x(x))(lambda x: x(x)))(ID)))

In [ ]:
# 変更例：最後の (ID) がないと、サンクが呼ばれず関数が返る
print(IF(TRUE)(lambda _: TRUE)(lambda _: FALSE))
print(to_bool(IF(TRUE)(lambda _: TRUE)(lambda _: FALSE)(ID)))
# 上のセルの IF(TRUE) を IF(FALSE) に変えると、止まらない側が選ばれて RecursionError

### 発展・演習：`XOR` を書く（スライド：「`XOR` を書く」）

ヒント：a に 2 択を選ばせ、`NOT` を使う。

In [ ]:
MY_XOR = None   # ← lambda a: lambda b: ... に書き換える

check(MY_XOR, [
    ("XOR(TRUE)(TRUE)", lambda XOR: to_bool(XOR(TRUE)(TRUE)), False),
    ("XOR(TRUE)(FALSE)", lambda XOR: to_bool(XOR(TRUE)(FALSE)), True),
    ("XOR(FALSE)(TRUE)", lambda XOR: to_bool(XOR(FALSE)(TRUE)), True),
    ("XOR(FALSE)(FALSE)", lambda XOR: to_bool(XOR(FALSE)(FALSE)), False),
])

In [ ]:
#@title 解答：XOR（スライド「`XOR` の解答」）（自分で考えてから開く）

XOR = lambda a: lambda b: a(NOT(b))(b)

print(to_bool(XOR(TRUE)(FALSE)), to_bool(XOR(TRUE)(TRUE)))
print(to_bool(XOR(FALSE)(TRUE)), to_bool(XOR(FALSE)(FALSE)))

## §4 チャーチ数（スライド 4 章）

数 n を「f を n 回適用する関数」で表す：$\overline{n} = \lambda f.\,\lambda x.\,f^n\,x$

### `ZERO`、`ONE` と変換関数 `to_int`（スライド：「チャーチ数：数 n を「f を n 回適用する関数」で表す」）

In [ ]:
ZERO = lambda f: lambda x: x
ONE = lambda f: lambda x: f(x)

to_int = lambda n: n(lambda x: x + 1)(0)

print(to_int(ZERO), to_int(ONE))

### `SUCC` と、`TEN` までの名前（スライド：「`SUCC`：f を適用する回数を 1 回増やす」）

In [ ]:
SUCC = lambda n: lambda f: lambda x: f(n(f)(x))
TWO = SUCC(ONE)
THREE = SUCC(TWO)
FOUR = SUCC(THREE)
FIVE = SUCC(FOUR)
SIX = SUCC(FIVE)
SEVEN = SUCC(SIX)
EIGHT = SUCC(SEVEN)
NINE = SUCC(EIGHT)
TEN = SUCC(NINE)

print(to_int(THREE), to_int(TEN))

In [ ]:
# 変更例：to_int の代わりに、f に文字列を包む関数、x に文字列を渡すと、f を何回適用したかが見える
print(THREE(lambda s: "f(" + s + ")")("x"))
print(ZERO(lambda s: "f(" + s + ")")("x"))

#### 実験用の道具 `to_church`（スライドには出てきません）

Python の整数からチャーチ数を作る、`to_int` と逆向きの変換で、ふつうの Python（`for`）で
`ZERO` に `SUCC` を k 回適用します。lambda で書く定義には使わず、実験の入力を作るときにだけ使います。

In [ ]:
def to_church(k):
    n = ZERO
    for _ in range(k):
        n = SUCC(n)
    return n


print(to_int(to_church(42)))

### 足し算と掛け算（スライド：「`ADD`：n に `SUCC` を m 回適用する」「`MUL`：関数合成 (composition) で書く」）

In [ ]:
ADD = lambda m: lambda n: m(SUCC)(n)
MUL_BY_ADD = lambda m: lambda n: m(ADD(n))(ZERO)   # 足し算の繰り返しで書いた掛け算（比較用）
MUL = lambda m: lambda n: lambda f: m(n(f))

print(to_int(ADD(TWO)(THREE)))
print(to_int(MUL_BY_ADD(TWO)(THREE)))
print(to_int(MUL(THREE)(FOUR)))

In [ ]:
# 変更例：MUL(TWO)(THREE) に文字列を包む関数を渡すと、f が 6 回適用されているのが見える
print(MUL(TWO)(THREE)(lambda s: "f(" + s + ")")("x"))

### 演習：`POW` を書く（スライド：「`POW` を書く（`EXP` とも）」）

`MY_POW = lambda m: lambda n: ＿＿＿`（$m^n$）を完成させよ。`MUL` より短く書けます。

ヒント：数 m 自身も「関数 f を受け取り、f の m 回合成を返す関数」なので、それを n 回繰り返すと？

In [ ]:
MY_POW = None   # ← lambda m: lambda n: ... に書き換える

check(MY_POW, [
    ("POW(TWO)(THREE)", lambda POW: to_int(POW(TWO)(THREE)), 8),
    ("POW(THREE)(TWO)", lambda POW: to_int(POW(THREE)(TWO)), 9),
    ("POW(FIVE)(ZERO)", lambda POW: to_int(POW(FIVE)(ZERO)), 1),
])

In [ ]:
#@title 解答：POW（スライド「`POW` の解答」）（自分で考えてから開く）

POW = lambda m: lambda n: n(m)

print(to_int(POW(TWO)(THREE)), to_int(POW(TWO)(TEN)))

### `IS_ZERO`（スライド：「`IS_ZERO`：n が f を 1 回でも適用すれば `FALSE`」）

In [ ]:
IS_ZERO = lambda n: n(lambda _: FALSE)(TRUE)

print(to_bool(IS_ZERO(ZERO)), to_bool(IS_ZERO(THREE)))

### 発展・演習：`IS_EVEN` を書く（スライド：「`IS_EVEN` を書く」）

ヒント：0 は偶数で、1 増えるたびに偶奇は反転する（真偽値を反転する関数は 3 章で作った）。

In [ ]:
MY_IS_EVEN = None   # ← lambda n: ... に書き換える

check(MY_IS_EVEN, [
    ("IS_EVEN(ZERO)", lambda IS_EVEN: to_bool(IS_EVEN(ZERO)), True),
    ("IS_EVEN(FOUR)", lambda IS_EVEN: to_bool(IS_EVEN(FOUR)), True),
    ("IS_EVEN(SEVEN)", lambda IS_EVEN: to_bool(IS_EVEN(SEVEN)), False),
])

In [ ]:
#@title 解答：IS_EVEN（スライド「`IS_EVEN` の解答」）（自分で考えてから開く）

IS_EVEN = lambda n: n(NOT)(TRUE)

print(to_bool(IS_EVEN(FOUR)), to_bool(IS_EVEN(SEVEN)))

## §5 ペアと引き算（スライド 5 章）

### ペア（スライド：「ペア：2 つの値を 1 つの関数にまとめる」）

In [ ]:
PAIR = lambda a: lambda b: lambda s: s(a)(b)
FIRST = lambda p: p(TRUE)
SECOND = lambda p: p(FALSE)

print(to_int(FIRST(PAIR(ONE)(TWO))), to_int(SECOND(PAIR(ONE)(TWO))))

In [ ]:
# 変更例：ペアに TRUE / FALSE 以外の関数を渡す
p = PAIR("左")("右")
print(repr(p(TRUE)), repr(p(FALSE)))
print(repr(p(lambda a: lambda b: b + a)))   # 観察用：2 つを逆順につなぐ

### Kleene のトリック：`SHIFT` と `PRED`（スライド：「前者関数 `PRED`：1 つ前の値もペアで持ち歩く」「`SHIFT` と `PRED`」）

ペア（前, 今）を持ち歩き、1 回ごとに今の値を前へずらし、今の値を +1 する：$(a, b) \mapsto (b, b+1)$

In [ ]:
SHIFT = lambda p: PAIR(SECOND(p))(SUCC(SECOND(p)))
PRED = lambda n: FIRST(n(SHIFT)(PAIR(ZERO)(ZERO)))

print(to_int(PRED(FIVE)), to_int(PRED(ZERO)))

In [ ]:
# （前, 今）のペアの変化を表示する（to_pair はペアを Python のタプルに変換する、このノートブック用の関数）
to_pair = lambda p: (to_int(FIRST(p)), to_int(SECOND(p)))

p = PAIR(ZERO)(ZERO)
for k in range(6):          # 変更例：回数を増やす
    print(k, "回:", to_pair(p))
    p = SHIFT(p)

### 発展・演習：`SUB` を書く（スライド：「`SUB` を書く」）

`MY_SUB(m)(n)` が m − n（m < n なら 0）になるように定義せよ。
ヒント：数 n は「何かを n 回適用する」関数なので、m に何を n 回適用すればよいか（4 章の ADD も参考になる）。

In [ ]:
MY_SUB = None   # ← lambda m: lambda n: ... に書き換える

check(MY_SUB, [
    ("SUB(SEVEN)(THREE)", lambda SUB: to_int(SUB(SEVEN)(THREE)), 4),
    ("SUB(TWO)(FIVE)", lambda SUB: to_int(SUB(TWO)(FIVE)), 0),
    ("SUB(FIVE)(ZERO)", lambda SUB: to_int(SUB(FIVE)(ZERO)), 5),
])

In [ ]:
#@title 解答：SUB（スライド「`SUB` の解答」。後の LEQ で使うので必ず実行）（自分で考えてから開く）

SUB = lambda m: lambda n: n(PRED)(m)

print(to_int(SUB(SEVEN)(THREE)), to_int(SUB(TWO)(FIVE)))

### 発展：大小比較 `LEQ` と `EQ`（スライド：「大小比較：`LEQ` と `EQ`」）

In [ ]:
LEQ = lambda m: lambda n: IS_ZERO(SUB(m)(n))
EQ = lambda m: lambda n: AND(LEQ(m)(n))(LEQ(n)(m))

print(to_bool(LEQ(THREE)(FIVE)), to_bool(EQ(FOUR)(FIVE)))

In [ ]:
# 変更例：遅さを測る（PRED は 1 回ごとに SHIFT を n 回、SUB はその PRED を n 回）
for k in [10, 20, 40]:
    n = to_church(k)
    start = time.perf_counter()
    to_bool(EQ(n)(n))
    print(k, f"{time.perf_counter() - start:.4f} 秒")

## §6 再帰と Y コンビネータ（スライド 6 章）

### 再帰定義は置き換えが終わらない（スライド：「再帰定義は置き換えが終わらない」）

下の `FACT_BAD` は右辺に自分の名前を含む再帰定義（比較用）で、
`FACT_BAD` を定義の右辺に置き換えると、その中にまた `FACT_BAD` が現れ、置き換えが終わりません。
Python で動くのは、呼び出した時点でグローバル名 `FACT_BAD` を探しに行く仕組みがあるからですが、ラムダ計算には名前の表がなく、名前は定義の右辺への置き換えでしかありません。

In [ ]:
FACT_BAD = lambda n: (          # 再帰定義：右辺に FACT_BAD 自身
    IF(IS_ZERO(n))
      (lambda _: ONE)
      (lambda _: MUL(n)(FACT_BAD(PRED(n))))
      (ID)
)

print(to_int(FACT_BAD(FIVE)), "← 動くが、Python のグローバル名の参照に頼っている")

### 自己適用（スライド：「自己適用：関数を自分自身に渡す」）

In [ ]:
omega = lambda x: x(x)

try:
    omega(omega)
except RecursionError as error:
    print("RecursionError:", error)

### `FACT_GEN`：`s(s)` で再帰する（スライド：「`FACT_GEN`：再帰の相手を引数 `s` で受け取る」「`FACT_GEN(FACT_GEN)` で階乗を計算する」）

In [ ]:
FACT_GEN = lambda s: lambda n: (
    IF(IS_ZERO(n))
      (lambda _: ONE)
      (lambda _: MUL(n)(s(s)(PRED(n))))
      (ID)
)

print(to_int(FACT_GEN(FACT_GEN)(THREE)), to_int(FACT_GEN(FACT_GEN)(FIVE)))

In [ ]:
# 変更例：s(s) を s に変えた場合
# s は「まだ s を待っている関数」なので、n を渡すと形が合わない
FACT_GEN_WRONG = lambda s: lambda n: (
    IF(IS_ZERO(n))
      (lambda _: ONE)
      (lambda _: MUL(n)(s(PRED(n))))
      (ID)
)
try:
    print(to_int(FACT_GEN_WRONG(FACT_GEN_WRONG)(THREE)))
except Exception as error:
    print(type(error).__name__ + ":", error)

### `FACT_BODY`：再帰の相手を引数で受け取る（スライド：「`s(s)` を本体の外へ出す」「`FACT_BODY`：再帰の相手を引数 `fact` で受け取る」）

再帰呼び出しの相手 `fact` を引数でもらうので、本物の階乗を `fact` に渡せば階乗になります。

In [ ]:
FACT_BODY = lambda fact: lambda n: (
    IF(IS_ZERO(n))
      (lambda _: ONE)
      (lambda _: MUL(n)(fact(PRED(n))))
      (ID)
)

print(to_int(FACT_BODY(FACT_GEN(FACT_GEN))(FIVE)))

In [ ]:
# 変更例：いつも 0 を返す偽の fact を渡すと、n = 0 のときだけ正しい結果になる
fake = lambda n: ZERO
print(to_int(FACT_BODY(fake)(ZERO)), to_int(FACT_BODY(fake)(THREE)))

### Python での Y コンビネータ（スライド：「FACT_GEN を組み立て直すと Y コンビネータになる」「Y F は F の不動点」「Python の Y は引数を先に評価して止まらない」）

$Y = \lambda f.\,(\lambda x.\,f\,(x\,x))\,(\lambda x.\,f\,(x\,x))$ の理屈は正しいのですが、
Python では数を渡す前に、`Y(FACT_BODY)` を **評価した時点で** RecursionError になります。

In [ ]:
Y = lambda f: (lambda x: f(x(x)))(lambda x: f(x(x)))

try:
    Y(FACT_BODY)
except RecursionError as error:
    print("RecursionError:", error)

### η展開と Z コンビネータ（スライド：「Z コンビネータ：Y の x x を η展開で包む」「`FACT = Z(FACT_BODY)`」）

In [ ]:
# 包むだけなら落ちない（中身は呼ばれるまで評価しない）
print(lambda v: omega(omega)(v))

In [ ]:
Z = lambda f: (
    (lambda x: f(lambda v: x(x)(v)))
    (lambda x: f(lambda v: x(x)(v)))
)
FACT = Z(FACT_BODY)

print(to_int(FACT(ZERO)), to_int(FACT(THREE)))
print(to_int(FACT(FIVE)))

In [ ]:
# 変更例：IF の枝をサンクで包まないと、n = 0 でも else 側を評価して止まらない
FACT_BODY_STRICT = lambda fact: lambda n: (
    IF(IS_ZERO(n))(ONE)(MUL(n)(fact(PRED(n)))))
try:
    print(to_int(Z(FACT_BODY_STRICT)(THREE)))
except RecursionError as error:
    print("RecursionError:", error)

### 演習：Z で `SUM_TO_N` を書く（スライド：「Z で `SUM_TO_N` を書く」）

0 + 1 + … + n を計算する関数を書く。ヒント：FACT_BODY をまねて、n = 0 なら ZERO、そうでなければ n + sum_to_n(n − 1)。
`...` の 2 か所を埋め、`MY_SUM_TO_N = None` の行を消して実行する。

In [ ]:
MY_SUM_TO_N = None   # ← 下のひな形を使うなら、この行を消す
# MY_SUM_TO_N = Z(lambda sum_to_n: lambda n: (
#     IF(IS_ZERO(n))
#       (lambda _: ...)
#       (lambda _: ...)
#       (ID)
# ))

check(MY_SUM_TO_N, [
    ("SUM_TO_N(ZERO)", lambda S: to_int(S(ZERO)), 0),
    ("SUM_TO_N(FOUR)", lambda S: to_int(S(FOUR)), 10),
    ("SUM_TO_N(TEN)", lambda S: to_int(S(TEN)), 55),
])

In [ ]:
#@title 解答：SUM_TO_N（スライド「`SUM_TO_N` の解答」）（自分で考えてから開く）

SUM_TO_N = Z(lambda sum_to_n: lambda n: (
    IF(IS_ZERO(n))
      (lambda _: ZERO)
      (lambda _: ADD(n)(sum_to_n(PRED(n))))
      (ID)
))

print(to_int(SUM_TO_N(FOUR)), to_int(SUM_TO_N(TEN)))

### 発展：`FIB`（スライド：「`FIB`：本体で自分を 2 回呼ぶ再帰」）

In [ ]:
FIB = Z(lambda fib: lambda n: (
    IF(IS_ZERO(PRED(n)))
      (lambda _: n)
      (lambda _: ADD(fib(PRED(n)))
                       (fib(PRED(PRED(n)))))
      (ID)
))

print(to_int(FIB(TEN)))

In [ ]:
# 変更例：n を変えて計算する（FACT は 8、FIB は 12 程度までなら短時間で終わる）
print([to_int(FACT(to_church(k))) for k in range(8)])
print([to_int(FIB(to_church(k))) for k in range(13)])

## §7 リストと FizzBuzz（スライド 7 章・発展）

5 章発展の `SUB`・`LEQ` を使います（このノートブックでは定義済み）。

### リストの表現（スライド：「リストをペアの入れ子で表す：`NIL`、`CONS`、`HEAD`、`TAIL`」）

外側のペアの 1 つ目は「空か？」を表す真偽値で、中身は 2 つ目のペアに入れます。

In [ ]:
NIL = PAIR(TRUE)(TRUE)
CONS = lambda h: lambda t: PAIR(FALSE)(PAIR(h)(t))
IS_NIL = FIRST
HEAD = lambda l: FIRST(SECOND(l))
TAIL = lambda l: SECOND(SECOND(l))

print(to_int(HEAD(CONS(ONE)(NIL))))
print(to_bool(IS_NIL(NIL)), to_bool(IS_NIL(CONS(ONE)(NIL))))

### 変換関数 `to_list`（スライド：「`to_list`：Python のリストに変換する」）

`def` と `while` を使いますが、結果の確認だけに使う変換関数なので可。

In [ ]:
def to_list(l, conv=lambda x: x):
    result = []
    while not to_bool(IS_NIL(l)):
        result.append(conv(HEAD(l)))
        l = TAIL(l)
    return result


print(to_list(CONS(ONE)(CONS(TWO)(NIL)), to_int))

### `RANGE`・`FOLD`・`MAP`（スライド：「`RANGE`：m から n までの数のリストを作る」「右畳み込み `FOLD`：`CONS` を `f` に、`NIL` を `z` に置き換える」「`MAP`：`FOLD` で書く」）

In [ ]:
RANGE = Z(lambda range_: lambda m: lambda n: (
    IF(LEQ(m)(n))
      (lambda _: CONS(m)(range_(SUCC(m))(n)))
      (lambda _: NIL)
      (ID)
))

print(to_list(RANGE(ONE)(FIVE), to_int))

In [ ]:
FOLD = Z(lambda fold: lambda f: lambda z: lambda l: (
    IF(IS_NIL(l))
      (lambda _: z)
      (lambda _: f(HEAD(l))(fold(f)(z)(TAIL(l))))
      (ID)
))

print(to_list(FOLD(CONS)(NIL)(RANGE(ONE)(THREE)), to_int))
print(to_int(FOLD(lambda _: SUCC)(ZERO)(RANGE(ONE)(FIVE))))   # 長さ

In [ ]:
# 変更例：FOLD の f と z に文字列の関数を渡すと、「CONS → f、NIL → z」の置き換えが見える
show = lambda x: lambda acc: "f(" + str(to_int(x)) + ")(" + acc + ")"
print(FOLD(show)("z")(RANGE(ONE)(THREE)))

In [ ]:
MAP = lambda f: lambda l: (
    FOLD(lambda x: lambda acc: CONS(f(x))(acc))(NIL)(l)
)

print(to_list(MAP(SUCC)(RANGE(ONE)(THREE)), to_int))

### 演習：`SUM` を `FOLD` で書く（スライド：「`SUM` を `FOLD` で書く」）

使ってよい：`FOLD`、`ADD`、`ZERO`。ヒント：CONS を何に、NIL を何に置き換える？

In [ ]:
MY_SUM = None   # ← ここに書く（1 行で書ける）

check(MY_SUM, [
    ("SUM(RANGE(ONE)(FOUR))", lambda SUM: to_int(SUM(RANGE(ONE)(FOUR))), 10),
    ("SUM(NIL)", lambda SUM: to_int(SUM(NIL)), 0),
])

In [ ]:
#@title 解答：SUM（スライド「`SUM` の解答：`FOLD` を使えば 1 行」）（自分で考えてから開く）

SUM = FOLD(ADD)(ZERO)

print(to_int(SUM(RANGE(ONE)(FOUR))))

### FizzBuzz で使う関数：`MOD`、`DIV`、`PUSH`、`TO_DIGITS`（スライド：「FizzBuzz を関数だけで書く」「`MOD`：m から n を引けるだけ引く」「`DIV`：m から n を引けた回数を数える」「`PUSH` と `TO_DIGITS`：数を各桁のリストにする」）

前提：除数 n は 1 以上で、n = ZERO は未定義です（この実装では止まりません）。

In [ ]:
MOD = Z(lambda mod: lambda m: lambda n: (
    IF(LEQ(n)(m))
      (lambda _: mod(SUB(m)(n))(n))
      (lambda _: m)
      (ID)
))
DIV = Z(lambda div: lambda m: lambda n: (
    IF(LEQ(n)(m))
      (lambda _: SUCC(div(SUB(m)(n))(n)))
      (lambda _: ZERO)
      (ID)
))

print(to_int(MOD(TEN)(THREE)), to_int(DIV(TEN)(THREE)))

In [ ]:
PUSH = lambda l: lambda x: FOLD(CONS)(CONS(x)(NIL))(l)
TO_DIGITS = Z(lambda digits: lambda n: (
    IF(LEQ(n)(NINE))
      (lambda _: CONS(n)(NIL))
      (lambda _: PUSH(digits(DIV(n)(TEN)))
                       (MOD(n)(TEN)))
      (ID)
))

print(to_list(TO_DIGITS(POW(TEN)(TWO)), to_int))

#### 追加の演習：`REVERSE`（スライドには出てきません）

リストを逆順にする `REVERSE` を `FOLD` と `PUSH` で定義せよ。
ヒント：右畳み込みは右端から処理するので、「残りを逆順にしたもの」の末尾に先頭を足せばよい。

In [ ]:
MY_REVERSE = None   # ← ここに書く

check(MY_REVERSE, [
    ("REVERSE(RANGE(ONE)(FOUR))",
     lambda R: to_list(R(RANGE(ONE)(FOUR)), to_int), [4, 3, 2, 1]),
    ("REVERSE(NIL)", lambda R: to_list(R(NIL), to_int), []),
])

In [ ]:
#@title 解答：REVERSE（自分で考えてから開く）

REVERSE = FOLD(lambda x: lambda acc: PUSH(acc)(x))(NIL)

print(to_list(REVERSE(RANGE(ONE)(FOUR)), to_int))

### 文字の表現と変換関数 `to_str`（スライド：「文字を 0〜14 の数で表す：`to_str`」）

| 数 | 0〜9 | 10 | 11 | 12 | 13 | 14 |
|---|---|---|---|---|---|---|
| 文字 | 0〜9 | B | F | i | u | z |

In [ ]:
ELEVEN = SUCC(TEN)
TWELVE = SUCC(ELEVEN)
THIRTEEN = SUCC(TWELVE)
FOURTEEN = SUCC(THIRTEEN)

to_str = lambda l: "".join(
    to_list(l, lambda c: "0123456789BFiuz"[to_int(c)])
)

print(repr(to_str(TO_DIGITS(FOURTEEN))))

### FizzBuzz を動かす（スライド：「`FIZZ`、`BUZZ`、`FIZZBUZZ_WORD`」「`FIZZBUZZ_ITEM`：数 n を FizzBuzz の 1 項目に変える」「`FIZZBUZZ`：1 から n までを変換して並べる」）

In [ ]:
FIZZ = CONS(ELEVEN)(
    CONS(TWELVE)(CONS(FOURTEEN)(CONS(FOURTEEN)(NIL)))
)
BUZZ = CONS(TEN)(
    CONS(THIRTEEN)(CONS(FOURTEEN)(CONS(FOURTEEN)(NIL)))
)
FIZZBUZZ_WORD = FOLD(CONS)(BUZZ)(FIZZ)

print(to_str(FIZZ), to_str(BUZZ), to_str(FIZZBUZZ_WORD))

In [ ]:
FIZZBUZZ_ITEM = lambda n: (
    IF(IS_ZERO(MOD(n)(THREE)))
      (lambda _: IF(IS_ZERO(MOD(n)(FIVE)))
          (lambda _: FIZZBUZZ_WORD)
          (lambda _: FIZZ)
          (ID))
      (lambda _: IF(IS_ZERO(MOD(n)(FIVE)))
          (lambda _: BUZZ)
          (lambda _: TO_DIGITS(n))
          (ID))
      (ID))
FIZZBUZZ = lambda n: MAP(FIZZBUZZ_ITEM)(RANGE(ONE)(n))
FIFTEEN = MUL(THREE)(FIVE)

print(to_list(FIZZBUZZ(FIFTEEN), to_str))

In [ ]:
# 変更例：範囲を広げる（再帰上限 1000 のままでも 200 程度までは動くが、時間がかかる）
start = time.perf_counter()
print(to_list(FIZZBUZZ(to_church(30)), to_str))
print(f"{time.perf_counter() - start:.3f} 秒")

## §8 SKI コンビネータ（スライド 8 章・発展）

### `S`、`K`、`I`（スライド：「コンビネータ `S`、`K`、`I`：`I` は `S K K`」）

$S = \lambda x.\,\lambda y.\,\lambda z.\,x\,z\,(y\,z)$、$K = \lambda x.\,\lambda y.\,x$、$I = S\,K\,K$

In [ ]:
S = lambda x: lambda y: lambda z: x(z)(y(z))
K = lambda x: lambda y: x
I = S(K)(K)

print(to_int(I(THREE)), to_bool(I(TRUE)))

### 演習：`K` と `K I` を簡約する（スライド：「`K` は `TRUE`、`K I` は `FALSE`」）

$K\,a\,b$ と $K\,I\,a\,b$ を紙の上で簡約してから、解答セルで確かめる。

In [ ]:
#@title 解答：K と K I（スライド「`K` は `TRUE`、`K I` は `FALSE`」）（自分で考えてから開く）

print(repr(K("a")("b")), repr(K(I)("a")("b")))
print(to_bool(K), to_bool(K(I)))   # K は TRUE、K I は FALSE のふるまい

### ブラケット抽象の例（スライド：「ブラケット抽象 (bracket abstraction)」「ブラケット抽象の例：ω と λx. SUCC x」）

$\mathcal{T}[\lambda x.\,x] = I$、$\mathcal{T}[\lambda x.\,E] = K\,\mathcal{T}[E]$（x が E に自由に出ない）、
$\mathcal{T}[\lambda x.\,E_1\,E_2] = S\,\mathcal{T}[\lambda x.\,E_1]\,\mathcal{T}[\lambda x.\,E_2]$

In [ ]:
print(to_int(S(I)(I)(ID)(THREE)))   # S I I は ω と同じふるまい
print(to_int(S(K(SUCC))(I)(TWO)))   # λx. SUCC x

try:
    S(I)(I)(S(I)(I))                # Ω と同じく止まらない
except RecursionError as error:
    print("RecursionError:", error)

#### 追加の演習：ブラケット抽象を手で行う（スライドには出てきません）

1. $\lambda x.\,\mathrm{NOT}\,x$ を S・K・I（と NOT）だけの式にして `MY_T1` に代入せよ。
2. $\lambda x.\,\lambda y.\,y\,x$を S・K・I だけの式にして `MY_T2` に代入せよ。内側の λ から変換する。

In [ ]:
MY_T1 = None   # ← S, K, I, NOT と適用だけで
MY_T2 = None   # ← S, K, I と適用だけで

check(MY_T1, [
    ("T1(TRUE)", lambda t: to_bool(t(TRUE)), False),
    ("T1(FALSE)", lambda t: to_bool(t(FALSE)), True),
])
check(MY_T2, [
    ('T2("a")(ID)', lambda t: t("a")(ID), "a"),
    ("T2(THREE)(SUCC)", lambda t: to_int(t(THREE)(SUCC)), 4),
])

In [ ]:
#@title 解答：ブラケット抽象（自分で考えてから開く）

# T[λx. NOT x] = S T[λx. NOT] T[λx. x] = S (K NOT) I
T1 = S(K(NOT))(I)
# T[λy. y x] = S I (K x)  →  T[λx. S I (K x)] = S (K (S I)) (S (K K) I)
T2 = S(K(S(I)))(S(K(K))(I))

print(to_bool(T1(TRUE)), repr(T2("a")(ID)), to_int(T2(THREE)(SUCC)))

## §9 まとめと先へ（スライド 9 章）

### 今日作ったもの（スライド：「今日作ったもの」）

冒頭で見せた計算で、数値リテラルも `if` も `def` も使っていません。

In [ ]:
print(to_int(FACT(FIVE)))

### 性能（スライド：「性能：チャーチ数の計算は遅く、スタックもすぐ溢れる」）

チャーチ数は一進法なので、`SUCC` の鎖で作った数は `to_int` で見るときにスタックが深くなり、
Python の再帰上限（既定 1000）を超えると `RecursionError` です。
Colab（Jupyter）ではカーネル自身もスタックを少し使うので、スライドの実測表より少し手前で溢れることがあります。

In [ ]:
print("再帰上限:", sys.getrecursionlimit())

for k in [100, 500, 2000]:     # 変更例：境界を探す
    try:
        print(k, "->", to_int(to_church(k)))
    except RecursionError:
        print(k, "-> RecursionError")

In [ ]:
# FACT の境界はスタックではなく時間で、1 つ増やすごとに大きく遅くなる
for k in range(6, 10):
    start = time.perf_counter()
    value = to_int(FACT(to_church(k)))
    print(k, value, f"{time.perf_counter() - start:.3f} 秒")

`sys.setrecursionlimit` で上限を上げることもできますが、上げすぎると Python 自体が落ちる（カーネルが再起動する）ことがあるので、上げる場合は少しずつ上げます。

### この先へ

- スライドと参照実装：[elle-et-noire/lambda-talk](https://github.com/elle-et-noire/lambda-talk)
  （[code/lambda_core.py](https://github.com/elle-et-noire/lambda-talk/blob/main/code/lambda_core.py) にこのノートブックと同じ定義がまとまっています）
- T. Stuart「[Programming with Nothing](https://tomstu.art/programming-with-nothing)」：Ruby の lambda（proc）だけで FizzBuzz まで書く講演と記事で、7 章はそのオマージュ
- 高橋正子『[計算論 計算可能性とラムダ計算](https://www.kindaikagaku.co.jp/book_list/detail/9784764901841/)』（近代科学社）、B. C. Pierce『[Types and Programming Languages](https://mitpress.mit.edu/9780262162098/types-and-programming-languages/)』（MIT Press）